# Tomato Leaf Disease — RRDN Baseline (PlantVillage)

This notebook implements a **paper-inspired RRDN baseline** for the tomato-only PlantVillage subset, designed for the first 50% milestone of the project.

### Project direction
- **Tomorrow:** reproduce a similar RRDN-style baseline, train/validate/test it, and generate evidence for the presentation.
- **Later:** integrate **CBAM** into the RRDN feature blocks and add **Grad-CAM** visual explanations.

> Important: the original paper reports an RRDN architecture and results, but does not provide an official public implementation. Therefore this notebook is an explicit **paper-faithful reconstruction**, not a claim that it is the authors' exact source code.


## 1. What the base paper actually specifies

The paper uses the AI Challenger 2018 tomato dataset: **13,185 images, 9 classes, 196×196 input**, with a **60/20/20 train/validation/test split**. It describes RRDN as a restructured residual-dense classifier, with batch normalization inside the RDB, a residual path from the original image, three pooling operations on that path, a flatten + dense classification head, cross-entropy loss, Adadelta with initial learning rate 1e-4, batch size 8, and 200 epochs. The paper reports 95% validation and test accuracy on its own AI Challenger setup.

For our experiment, the dataset is changed to **PlantVillage tomato**, so the number of classes and class distribution will be determined from the actual downloaded dataset rather than copied from the paper.


## 2. PlantVillage choice

The selected Kaggle dataset is the PlantVillage dataset. The tomato subset is commonly represented by 10 classes: bacterial spot, early blight, late blight, leaf mold, Septoria leaf spot, spider mites, target spot, tomato mosaic virus, tomato yellow leaf curl virus, and healthy.

**Do not hard-code the image counts.** Different published copies/versions of PlantVillage report slightly different counts, especially for the yellow-leaf-curl-virus class. This notebook scans the downloaded directory and prints the exact counts in the copy you are using before training.


## 3. Runtime / persistence strategy

1. Mount Google Drive.
2. Keep the dataset cache, train/validation/test split CSV, checkpoints, logs, class names, and final model in Drive.
3. Optionally copy the tomato subset from Drive to `/content` for faster GPU training. The persistent copy remains in Drive.
4. If the Colab runtime disconnects, remount Drive and reload the saved `.keras` checkpoint.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/Tomato_RRDN_Project')
DATA_ROOT = PROJECT_ROOT / 'data'
SPLIT_ROOT = PROJECT_ROOT / 'splits'
MODEL_ROOT = PROJECT_ROOT / 'models'
LOG_ROOT = PROJECT_ROOT / 'logs'
RESULT_ROOT = PROJECT_ROOT / 'results'

for p in [DATA_ROOT, SPLIT_ROOT, MODEL_ROOT, LOG_ROOT, RESULT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('Persistent project directory:', PROJECT_ROOT)


Mounted at /content/drive
Persistent project directory: /content/drive/MyDrive/Tomato_RRDN_Project


In [5]:
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/Tomato_RRDN_Project')

DATA_ROOT = PROJECT_ROOT / 'data'
SPLIT_ROOT = PROJECT_ROOT / 'splits'
MODEL_ROOT = PROJECT_ROOT / 'models'
LOG_ROOT = PROJECT_ROOT / 'logs'
RESULT_ROOT = PROJECT_ROOT / 'results'

for p in [DATA_ROOT, SPLIT_ROOT, MODEL_ROOT, LOG_ROOT, RESULT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# Dataset uploaded manually to Google Drive
dataset_path = DATA_ROOT / 'plantvillage'

print("Dataset path:", dataset_path)
print("Exists:", dataset_path.exists())
print("Contents:", list(dataset_path.iterdir()))

Dataset path: /content/drive/MyDrive/Tomato_RRDN_Project/data/plantvillage
Exists: True
Contents: [PosixPath('/content/drive/MyDrive/Tomato_RRDN_Project/data/plantvillage/color')]


In [2]:
!nvidia-smi

import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

assert tf.config.list_physical_devices('GPU'), 'GPU not detected. In Colab use Runtime > Change runtime type > GPU.'


Mon Sep 28 05:14:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 5. Locate the tomato class directory automatically

Kaggle copies can contain an extra `plantvillage dataset/color/` level. The code below does not assume one exact folder layout.

In [ ]:
color_root = dataset_path / "color"

if not color_root.exists():
    raise FileNotFoundError(
        f"Could not find color directory at: {color_root}"
    )

class_dirs = sorted(
    [
        p for p in color_root.iterdir()
        if p.is_dir() and p.name.startswith("Tomato_")
    ],
    key=lambda p: p.name.lower()
)

print("Dataset root:", color_root)
print("Classes found:", len(class_dirs))

for d in class_dirs:
    count = sum(
        1
        for p in d.iterdir()
        if p.is_file()
        and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )

    print(f"{d.name:45s} {count:6d}")

assert len(class_dirs) == 10, (
    f"Expected 10 tomato classes, found {len(class_dirs)}"
)

## 6. Create a persistent deterministic 60/20/20 split

The base paper uses 60% training, 20% validation and 20% test. We keep that split for comparability, while using PlantVillage instead of AI Challenger.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

split_csv = SPLIT_ROOT / 'tomato_split.csv'

records = []
class_names = [d.name for d in class_dirs]
class_to_idx = {name: i for i, name in enumerate(class_names)}

for d in class_dirs:
    for p in d.iterdir():
        if p.is_file() and p.suffix in IMAGE_EXTS:
            records.append({'path': str(p), 'class_name': d.name, 'label': class_to_idx[d.name]})

df = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)

if split_csv.exists():
    split_df = pd.read_csv(split_csv)
    print('Loaded existing persistent split:', split_csv)
else:
    train_df, temp_df = train_test_split(
        df, test_size=0.40, random_state=42, stratify=df['label']
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, random_state=42, stratify=temp_df['label']
    )
    train_df = train_df.assign(split='train')
    val_df = val_df.assign(split='val')
    test_df = test_df.assign(split='test')
    split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    split_df.to_csv(split_csv, index=False)
    print('Created persistent split:', split_csv)

print(split_df['split'].value_counts())
print('\nClass distribution:')
print(pd.crosstab(split_df['class_name'], split_df['split']))


## 7. Training configuration

The important paper-aligned settings are retained where practical: **196×196 input, Adadelta, learning rate 1e-4, cross-entropy, batch size 8**. For tomorrow's milestone, the default maximum is reduced from the paper's 200 epochs to 50 with early stopping, so we can obtain a validated result within a realistic Colab session. If time permits, increase `EPOCHS` later.


In [ ]:
IMG_SIZE = (196, 196)
BATCH_SIZE = 8
EPOCHS = 50
LEARNING_RATE = 1e-4
SEED = 42
NUM_CLASSES = len(class_names)

print('NUM_CLASSES =', NUM_CLASSES)
print('class_names =', class_names)


## 8. TensorFlow input pipeline

Images are resized to 196×196 and normalized to [0, 1]. No aggressive augmentation is applied in the baseline, keeping this first experiment closer to the paper's described training procedure.

In [ ]:
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE

def decode_resize(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE, antialias=True)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def make_dataset(part_df, shuffle=False):
    paths = part_df['path'].astype(str).values
    labels = part_df['label'].astype('int32').values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(part_df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_resize, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_df = split_df[split_df['split'] == 'train'].reset_index(drop=True)
val_df = split_df[split_df['split'] == 'val'].reset_index(drop=True)
test_df = split_df[split_df['split'] == 'test'].reset_index(drop=True)

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

print('train:', len(train_df), 'val:', len(val_df), 'test:', len(test_df))


## 9. RRDN reconstruction

The implementation follows the paper's described ideas and Figure 7: an RDB-style feature extractor with three convolutional layers, batch normalization after the RDB, a residual/dense concatenation, a downsampled original-image branch, another residual branch, three pooling operations on the original-image path, and a flatten + softmax classification head.

**This is the important scientific wording for the presentation:** call it a *reconstructed/paper-inspired RRDN baseline*, because the authors' original implementation is not publicly available.

In [ ]:
from tensorflow.keras import layers, Model, regularizers

def rdb_block(x, filters=64, name='rdb'):
    y = layers.Conv2D(filters, 3, padding='same', name=f'{name}_conv1')(x)
    y = layers.ReLU(name=f'{name}_relu1')(y)
    y = layers.Conv2D(filters, 3, padding='same', name=f'{name}_conv2')(y)
    y = layers.ReLU(name=f'{name}_relu2')(y)
    y = layers.Conv2D(filters, 3, padding='same', name=f'{name}_conv3')(y)
    y = layers.ReLU(name=f'{name}_relu3')(y)
    y = layers.BatchNormalization(name=f'{name}_bn')(y)
    y = layers.LeakyReLU(negative_slope=0.3, name=f'{name}_leakyrelu')(y)
    return y

def build_rrdn(num_classes):
    inputs = layers.Input(shape=(*IMG_SIZE, 3), name='input_image')

    # Initial feature extraction: 196x196x3 -> 196x196x64
    stem = layers.Conv2D(64, 3, padding='same', name='stem_conv')(inputs)
    stem = layers.ReLU(name='stem_relu')(stem)

    # Main RRDN/RDB path
    x = layers.Conv2D(64, 3, padding='same', name='pre_rdb_conv')(stem)
    x = layers.ReLU(name='pre_rdb_relu')(x)

    rdb = rdb_block(x, 64, name='rdb1')
    x = layers.Concatenate(name='rdb_residual_concat')([rdb, stem])
    x = layers.Conv2D(64, 3, padding='same', name='feature_fusion_conv')(x)
    x = layers.BatchNormalization(name='feature_fusion_bn')(x)

    # Part-C-like residual branch: downsample early features
    branch_c = layers.MaxPooling2D(pool_size=3, strides=2, padding='same', name='branch_c_pool')(stem)

    # First downsampling + residual merge
    x = layers.MaxPooling2D(pool_size=2, strides=2, padding='same', name='main_pool1')(x)
    x = layers.Add(name='residual_add1')([x, branch_c])

    x = layers.Conv2D(128, 3, padding='same', name='post_res_conv')(x)
    x = layers.ReLU(name='post_res_relu')(x)
    x = layers.MaxPooling2D(pool_size=2, strides=2, padding='same', name='main_pool2')(x)
    x = layers.Conv2D(128, 3, padding='same', name='deep_conv')(x)
    x = layers.ReLU(name='deep_relu')(x)
    x = layers.MaxPooling2D(pool_size=2, strides=2, padding='same', name='main_pool3')(x)

    # Part-B-like original image residual path: three pooling operations
    branch_b = layers.Conv2D(128, 3, padding='same', name='original_image_conv')(inputs)
    branch_b = layers.ReLU(name='original_image_relu')(branch_b)
    branch_b = layers.MaxPooling2D(2, 2, padding='same', name='branch_b_pool1')(branch_b)
    branch_b = layers.MaxPooling2D(2, 2, padding='same', name='branch_b_pool2')(branch_b)
    branch_b = layers.MaxPooling2D(2, 2, padding='same', name='branch_b_pool3')(branch_b)

    # Target layer for future Grad-CAM use
    x = layers.Add(name='gradcam_target')([x, branch_b])
    x = layers.Flatten(name='flatten')(x)
    outputs = layers.Dense(
        num_classes,
        activation='softmax',
        kernel_regularizer=regularizers.l2(1e-4),
        name='classifier'
    )(x)

    return Model(inputs, outputs, name='RRDN_PlantVillage_Baseline')

model = build_rrdn(NUM_CLASSES)
model.summary()


### Sanity-check the output shape before training

In [ ]:
sample_images, sample_labels = next(iter(train_ds.take(1)))
sample_logits = model(sample_images, training=False)
print('Input batch:', sample_images.shape)
print('Output batch:', sample_logits.shape)
assert sample_logits.shape[-1] == NUM_CLASSES


## 10. Compile and train

Paper-aligned optimizer: **Adadelta**, initial learning rate **1e-4**. We use early stopping and save the best validation-loss checkpoint to Drive.

In [ ]:
import numpy as np

model.compile(
    optimizer=tf.keras.optimizers.Adadelta(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')]
)

BEST_MODEL = MODEL_ROOT / 'best_rrdn_baseline.keras'
HISTORY_CSV = LOG_ROOT / 'rrdn_training.csv'

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL),
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(str(HISTORY_CSV), append=True),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=8, restore_best_weights=True, verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)


## 11. Training curves

In [ ]:
import matplotlib.pyplot as plt

hist = pd.DataFrame(history.history)

plt.figure(figsize=(8,5))
plt.plot(hist['accuracy'], label='train accuracy')
plt.plot(hist['val_accuracy'], label='validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('RRDN Baseline — Accuracy')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8,5))
plt.plot(hist['loss'], label='train loss')
plt.plot(hist['val_loss'], label='validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('RRDN Baseline — Loss')
plt.legend()
plt.grid(True)
plt.show()


## 12. Test-set evaluation

The test set is untouched during training. Report accuracy, macro precision/recall/F1, per-class metrics, and the confusion matrix.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns

if BEST_MODEL.exists():
    model = tf.keras.models.load_model(BEST_MODEL)
    print('Loaded best checkpoint:', BEST_MODEL)

test_loss, test_acc = model.evaluate(test_ds, verbose=1)

y_true = []
y_pred = []

for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(f'\nTest accuracy: {accuracy_score(y_true, y_pred):.4f}')
print('\nClassification report:')
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
pd.DataFrame(report).T.to_csv(RESULT_ROOT / 'classification_report.csv')

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('RRDN Baseline — Test Confusion Matrix')
plt.xticks(rotation=60, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(RESULT_ROOT / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()


## 13. Show qualitative predictions

These examples are useful for tomorrow's presentation: show the original leaf, true class, predicted class, and confidence.

In [ ]:
def show_predictions(dataset, n=12):
    images, labels = next(iter(dataset.unbatch().batch(n)))
    probs = model.predict(images, verbose=0)
    preds = np.argmax(probs, axis=1)
    confs = np.max(probs, axis=1)

    cols = 4
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=(16, 4 * rows))
    for i in range(n):
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(images[i].numpy())
        ax.axis('off')
        true_name = class_names[int(labels[i])]
        pred_name = class_names[int(preds[i])]
        ax.set_title(f'True: {true_name}\nPred: {pred_name}\nConf: {confs[i]:.2%}')
    plt.tight_layout()
    plt.show()

show_predictions(test_ds, n=min(12, len(test_df)))


## 14. Save experiment metadata


In [ ]:
import json

metadata = {
    'model': 'RRDN paper-inspired reconstruction',
    'dataset': 'PlantVillage tomato subset',
    'image_size': list(IMG_SIZE),
    'num_classes': NUM_CLASSES,
    'class_names': class_names,
    'split': '60/20/20 stratified',
    'batch_size': BATCH_SIZE,
    'epochs_max': EPOCHS,
    'optimizer': 'Adadelta',
    'learning_rate': LEARNING_RATE,
    'seed': SEED,
    'best_model_path': str(BEST_MODEL),
    'split_csv': str(split_csv)
}

with open(RESULT_ROOT / 'experiment_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))


## 15. Later milestone — CBAM integration

Do **not** run this tomorrow unless the baseline is already working.

CBAM should be inserted into the RRDN feature path after an RDB/fused feature tensor. CBAM first learns **channel attention** (what features are useful) and then **spatial attention** (where useful information is located). The original CBAM paper describes the sequential channel-then-spatial arrangement.

The key experiment will eventually be:

| Model | Same split | Accuracy | Macro F1 | Explainability |
|---|---|---:|---:|---|
| RRDN baseline | Yes | measured | measured | optional |
| RRDN + CBAM | Yes | measured | measured | Grad-CAM |

This makes the effect of CBAM measurable rather than simply claiming improvement.

## 16. Later milestone — Grad-CAM

The model already names `gradcam_target` so a Grad-CAM implementation can target a final convolutional feature tensor later. Grad-CAM uses the gradients flowing into a target convolutional layer to produce a class-discriminative localization heatmap.

For the final project, generate explanations for correct and incorrect predictions and discuss whether the heatmap overlaps the diseased leaf region rather than the background.

## Presentation checklist for tomorrow

- [ ] Show the PlantVillage tomato classes and class counts from the actual run.
- [ ] Show the 60/20/20 split.
- [ ] Show the reconstructed RRDN architecture/model summary.
- [ ] Show training vs validation accuracy/loss.
- [ ] Report test accuracy.
- [ ] Show precision/recall/F1 and confusion matrix.
- [ ] Show 6–12 qualitative predictions.
- [ ] Clearly state: **baseline is a reconstruction of the paper's described RRDN architecture; the original authors' implementation is not being claimed.**
- [ ] Do not claim the paper's 95% result as your own result. That 95% belongs to the paper's AI Challenger experiment.
